# 04 – Gerar Relatório PDF  

Este notebook consolida as análises em um relatório PDF contendo:
 - Tabelas dos 10 principais países (verão, inverno, total)
 - Gráficos gerados no notebook 03 (com rótulos e tradução)
 
Os arquivos de saída são salvos em `gold/relatorio/`.


In [33]:
import pandas as pd
from pathlib import Path
from fpdf import FPDF
from datetime import date
import matplotlib.pyplot as plt
import country_converter as coco
import unicodedata
from PIL import Image


# Caminhos
BRONZE_DIR = Path("../bronze")
GOLD_DIR = Path("../gold/analise_medalhas")
REPORT_DIR = Path("../gold/relatorio")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [34]:
# Carregar dados unificados
df = pd.read_parquet(BRONZE_DIR / "medalhas_unificado.parquet")
df_paris = pd.read_parquet(BRONZE_DIR / "olympics_paris2024.parquet")


In [35]:
# Tradução de países (usando country_converter)
cc = coco.CountryConverter()
def traduzir_pais(nome):
    try:
        pt = cc.convert(names=nome, to='name_pt')
        return pt if pt and pt != nome else nome
    except:
        return nome

def medal_table(df, season=None):
    if season is not None:
        df = df[df['season'] == season]
    agg = df.groupby('country')[['gold', 'silver', 'bronze']].sum().reset_index()
    agg['total'] = agg['gold'] + agg['silver'] + agg['bronze']
    agg = agg.sort_values('total', ascending=False).reset_index(drop=True)
    agg['country_pt'] = agg['country'].apply(traduzir_pais)
    return agg

# Gerar tabelas
summer = medal_table(df, 'Summer')
winter = medal_table(df, 'Winter')
total = medal_table(df)

# Função para remover acentos e caracteres não ASCII (compatível com latin1)
def ascii_only(text):
    if not isinstance(text, str):
        text = str(text)
    # Remove acentos
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')
    # Substituir caracteres especiais
    replacements = {
        '\u2013': '-',   # en dash
        '\u2014': '-',   # em dash
        '\u2018': "'",   # left single quote
        '\u2019': "'",   # right single quote
        '\u201c': '"',   # left double quote
        '\u201d': '"',   # right double quote
        '\u2026': '...', # ellipsis
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


In [36]:
# Definição do PDF com margens personalizadas
class PDF(FPDF):
    def __init__(self):
        super().__init__()
        # Margens: esquerda, topo, direita, inferior (em mm)
        self.set_margins(20, 25, 20)
        self.set_auto_page_break(auto=True, margin=20)
        # Usar fonte padrão embutida (Helvetica)
        self.set_font('Helvetica', '', 10)

    def header(self):
        # Linha decorativa superior
        self.set_y(15)
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 10, ascii_only('Relatorio de Medalhas Olimpicas (1896-2024)'), 0, 1, 'C')
        self.set_y(22)
        self.set_draw_color(200, 200, 200)
        self.line(20, 25, 190, 25)  # linha horizontal (x1, y1, x2, y2)
        self.ln(10)

    def footer(self):
        self.set_y(-20)
        self.set_font('Helvetica', 'I', 8)
        self.set_draw_color(200, 200, 200)
        self.line(20, self.get_y(), 190, self.get_y())
        self.cell(0, 10, ascii_only(f'Pagina {self.page_no()}'), 0, 0, 'C')

    def add_section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_text_color(50, 50, 50)
        self.cell(0, 10, ascii_only(title), 0, 1, 'L')
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def add_table(self, data, headers, col_widths, highlight_top=10):
        """
        Adiciona tabela colorida com cabeçalho escuro e linhas alternadas.
        """
        # Cabeçalho
        self.set_font('Helvetica', 'B', 9)
        self.set_fill_color(60, 60, 60)  # cinza escuro
        self.set_text_color(255, 255, 255)
        for i, header in enumerate(headers):
            self.cell(col_widths[i], 8, ascii_only(header), 1, 0, 'C', 1)
        self.ln()
        self.set_text_color(0, 0, 0)
        self.set_font('Helvetica', '', 9)

        # Corpo
        for idx, row in data.iterrows():
            # Alternar cor de fundo
            if idx % 2 == 0:
                self.set_fill_color(245, 245, 245)  # cinza claro
            else:
                self.set_fill_color(255, 255, 255)  # branco
            # Primeira coluna (país)
            self.cell(col_widths[0], 7, ascii_only(row['country_pt']), 1, 0, 'L', 1)
            # Colunas de medalhas
            self.cell(col_widths[1], 7, str(row['gold']), 1, 0, 'C', 1)
            self.cell(col_widths[2], 7, str(row['silver']), 1, 0, 'C', 1)
            self.cell(col_widths[3], 7, str(row['bronze']), 1, 0, 'C', 1)
            self.cell(col_widths[4], 7, str(row['total']), 1, 0, 'C', 1)
            self.ln()

        self.ln(5)

    def add_image_resized(self, img_path, max_width=170):
        """Adiciona imagem redimensionada mantendo proporção e centralizada."""
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                # Calcular altura proporcional
                ratio = max_width / w
                new_w = max_width
                new_h = h * ratio
                # Centralizar horizontalmente
                x = (self.w - new_w) / 2
                self.image(str(img_path), x=x, y=self.get_y(), w=new_w, h=new_h)
                self.ln(new_h + 5)
        except Exception as e:
            self.cell(0, 10, ascii_only(f'Erro ao carregar imagem: {img_path.name}'), 0, 1, 'C')
            print(f"Erro: {e}")


In [37]:
# Criar PDF
pdf = PDF()
pdf.add_page()

# Título principal (já no header, mas pode adicionar um subtítulo)
pdf.set_y(40)  # após o header
pdf.set_font('Helvetica', 'B', 14)
pdf.cell(0, 10, ascii_only('Analise Completa de Medalhas Olimpicas'), 0, 1, 'C')
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 10, ascii_only(f'Gerado em {date.today().strftime("%d/%m/%Y")}'), 0, 1, 'C')
pdf.ln(10)

# Seção: Top 10 Verão
pdf.add_section_title('Top 10 – Jogos de Verao')
col_widths = [60, 25, 25, 25, 25]
headers = ['Pais', 'Ouro', 'Prata', 'Bronze', 'Total']
pdf.add_table(summer.head(10), headers, col_widths)

# Seção: Top 10 Inverno
pdf.add_section_title('Top 10 – Jogos de Inverno')
pdf.add_table(winter.head(10), headers, col_widths)

# Seção: Top 10 Total
pdf.add_section_title('Top 10 – Total Geral')
pdf.add_table(total.head(10), headers, col_widths)

# Gráficos (cada um em nova página para melhor visualização)
graphs = [
    ('medalhas_plot.png', 'Gráfico 1: Top 50 Países Mais Medalhados (Verao, Inverno, Total)'),
    ('evolucao_medalhas.png', 'Gráfico 2: Evolucao do Total de Medalhas por Ano'),
    ('distribuicao_medalhas.png', 'Gráfico 3: Distribuicao de Medalhas por Tipo'),
    ('medalhas_por_genero.png', 'Gráfico 4: Medalhas por Genero em Paris 2024')
]

for img_name, caption in graphs:
    pdf.add_page()
    pdf.add_section_title(caption)
    img_path = GOLD_DIR / img_name
    if img_path.exists():
        pdf.add_image_resized(img_path, max_width=170)
    else:
        pdf.cell(0, 10, ascii_only(f'Grafico {img_name} nao encontrado.'), 0, 1, 'C')

# Salvar PDF
pdf_output = REPORT_DIR / "relatorio_medalhas.pdf"
pdf.output(str(pdf_output))
print(f"PDF salvo em {pdf_output}")

PDF salvo em ../gold/relatorio/relatorio_medalhas.pdf
